In [1]:
import pandas as pd
from datetime import datetime
from numpy import nan
from pandas import DataFrame

In [2]:
def get_nba_season(game_date):
    """Determine the NBA season for a given date.
    NBA season spans October-June, labeled as YYYY/YY where October is in the first year.
    Example: Feb 25, 2025 → season "2024/25" (because season started Oct 2024)"""
    year = game_date.year
    month = game_date.month

    # If month is October or later, season is current_year/next_year
    if month >= 9:
        return f"{year}/{(year + 1) % 100:02d}"
    # Otherwise season is previous_year/current_year
    else:
        return f"{year - 1}/{year % 100:02d}"

def is_home(home_game, away_game):
    if home_game == 1:
        return 1
    elif away_game == 1:
        return 0
    else:
        return nan

def is_away(home_game, away_game):
    if home_game == 1:
        return 0
    elif away_game == 1:
        return 1
    else:
        return nan
    
def read_games_file() -> DataFrame:
    # Read the CSV file directly into a pandas DataFrame
    file_path = "data/archive/Games.csv"  # Change to your actual filename
    df = pd.read_csv(file_path, parse_dates=["gameDate"])

    # Convert specific columns to appropriate data types
    df["gameId"] = df["gameId"].astype("int64")
    df["hometeamId"] = df["hometeamId"].astype("int64")
    df["awayteamId"] = df["awayteamId"].astype("int64")
    df["winner"] = df["winner"].astype("int64")
    df["arenaId"] = df["arenaId"].astype("int64")
    df["seriesGameNumber"] = df["seriesGameNumber"].astype("float64")

    # Handle empty strings in numeric columns (convert to NaN)
    numeric_cols = ["homeScore", "awayScore", "attendance"]
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


def make_rested_days_table(
    games_filtered, start_date, end_date, teams_season
) -> DataFrame:
    date_range = pd.date_range(start=start_date, end=end_date)
    rested_days = pd.MultiIndex.from_product(
        [date_range, teams_season], names=["gameDate", "teamId"]
    ).to_frame(index=False)
    rested_days["gameDateOnlyStr"] = rested_days["gameDate"].dt.strftime("%Y-%m-%d")

    rested_days = rested_days.merge(
        games_filtered[["gameDateOnlyStr", "hometeamId"]],
        left_on=["gameDateOnlyStr", "teamId"],
        right_on=["gameDateOnlyStr", "hometeamId"],
        how="left",
        indicator="home_game",
    ).drop("hometeamId", axis=1)

    rested_days = rested_days.merge(
        games_filtered[["gameDateOnlyStr", "awayteamId"]],
        left_on=["gameDateOnlyStr", "teamId"],
        right_on=["gameDateOnlyStr", "awayteamId"],
        how="left",
        indicator="away_game",
    ).drop("awayteamId", axis=1)

    rested_days["home_game"] = rested_days["home_game"].apply(
        lambda x: 1 if x == "both" else 0
    )

    rested_days["away_game"] = rested_days["away_game"].apply(
        lambda x: 1 if x == "both" else 0
    )

    rested_days["rest"] = rested_days.apply(
        lambda x: abs((x["home_game"] | x["away_game"]) - 1), axis=1
    )

    rested_days = rested_days.sort_values(["teamId", "gameDate"])
    rested_days["rest"] = (
        rested_days.groupby("teamId")["rest"].shift(1).fillna(1).astype(int)
    )

    rested_days["rested_days"] = rested_days.groupby(
        ["teamId", (rested_days["rest"] == 0).cumsum()]
    )["rest"].transform("cumsum")

    # At Home Count
    rested_days["at_home_indicator"] = rested_days.apply(
        lambda x: is_home(x["home_game"], x["away_game"]), axis=1
    )
    rested_days["at_home_indicator"] = (
        rested_days.groupby("teamId")["at_home_indicator"].ffill().fillna(1).astype(int)
    )
    rested_days["days_at_home"] = rested_days.groupby(
        ["teamId", (rested_days["at_home_indicator"] == 0).cumsum()]
    )["at_home_indicator"].transform("cumsum")

    # On the Road Count
    rested_days["at_road_indicator"] = rested_days.apply(
        lambda x: is_away(x["home_game"], x["away_game"]), axis=1
    )
    rested_days["at_road_indicator"] = (
        rested_days.groupby("teamId")["at_road_indicator"].ffill().fillna(0).astype(int)
    )
    rested_days["days_on_road"] = rested_days.groupby(
        ["teamId", (rested_days["at_road_indicator"] == 0).cumsum()]
    )["at_road_indicator"].transform("cumsum")

    return rested_days


In [3]:
def add_features_to_games(df):
    df["season"] = df["gameDate"].apply(get_nba_season)

    df["winnerTeam"] = df.apply(
        lambda x: str(x["hometeamName"])
        if x["winner"] == x["hometeamId"]
        else str(x["awayteamName"])
        if x["winner"] == x["awayteamId"]
        else "",
        axis=1,
    )

    df["gameDateOnlyStr"] = df["gameDate"].dt.strftime("%Y-%m-%d")

    df["pts_diff"] = df.apply(
        lambda x: x["homeScore"] - x["awayScore"],
        axis=1,
    )

    df["winner_home_bool"] = df.apply(
        lambda x: 1 if x["winner"] == x["hometeamId"] else 0, axis=1
    )

    df["winner_away_bool"] = df.apply(
        lambda x: 1 if x["winner"] != x["hometeamId"] else 0, axis=1
    )

    return df

In [4]:
def filter_games(games, start_date, gameType) -> DataFrame:
    games = games.loc[games["gameType"] <= gameType]
    games = games.loc[games["gameDate"] >= start_date]
    games = games.loc[games["homeScore"] > 0]

    return games

In [5]:
gameType = "Regular Season"
start_date = "1980-07-01"

games = read_games_file()
games = add_features_to_games(games)
games_filtered: DataFrame = filter_games(games, start_date, gameType)

# Get seasons infos
season_teams_ids = games_filtered.groupby("season")["hometeamId"].unique().reset_index()
sesaon_start_end = (
    games_filtered.groupby("season")["gameDate"].agg(["min", "max"]).reset_index()
)
season_info = sesaon_start_end.merge(season_teams_ids, on="season")
season_info_list = season_info.to_dict(orient="records")

# Get seasons tables
seasons_dict = {}
for season_info in season_info_list:
    start_date = season_info["min"]
    end_date = season_info["max"]
    teams_season = season_info["hometeamId"]

    seasons_dict[season_info["season"]] = {"rested_days": make_rested_days_table(
        games_filtered, start_date, end_date, teams_season
    )}

/var/folders/_2/8lt451812jdgdwl3yhp1bbhh0000gn/T/ipykernel_79384/2281226083.py:34: DtypeWarning: Columns (15) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path, parse_dates=["gameDate"])


In [7]:
seasons_dict["1996/97"]["rested_days"] 


,gameDate,teamId,gameDateOnlyStr,home_game,away_game,rest,rested_days,at_home_indicator,days_at_home,at_road_indicator,days_on_road
6,1996-11-01 19:00:00,1610612737,1996-11-01,0,1,1,1,0,0,1,1
35,1996-11-02 19:00:00,1610612737,1996-11-02,1,0,0,0,1,1,0,0
64,1996-11-03 19:00:00,1610612737,1996-11-03,0,0,0,0,1,2,0,0
93,1996-11-04 19:00:00,1610612737,1996-11-04,0,1,1,1,0,0,1,1
122,1996-11-05 19:00:00,1610612737,1996-11-05,0,1,0,0,0,0,1,2
...,...,...,...,...,...,...,...,...,...,...,...
6395,1997-06-09 19:00:00,1610612766,1997-06-09,0,0,1,41,1,43,0,0
6424,1997-06-10 19:00:00,1610612766,1997-06-10,0,0,1,42,1,44,0,0
6453,1997-06-11 19:00:00,1610612766,1997-06-11,0,0,1,43,1,45,0,0
6482,1997-06-12 19:00:00,1610612766,1997-06-12,0,0,1,44,1,46,0,0


In [ ]:
# rested_days[rested_days["teamId"] == 1610612743].head(20)

In [ ]:
# set(rested_days.teamId)